# Spacecraft power budget
**Start:** select **Kernel → Restart & Run All**. In **Edit variants**, click **Create variant**, edit inputs, then **Save & compare**. The comparison includes Baseline and your saved variant automatically.

**Compare:** tick variants, choose nominal/safe mode and outputs, then **Update comparison**. The table shows results side by side. The array-area explanation shows the calculation drivers; equal areas are valid when their power-to-output ratios match. Peak-only and battery-only changes do not change array area.

**Sweep:** choose variants and an input, enter a range or comma-separated values, review the optional links, then **Run sweep**. Percentages use 0–100. Battery count ranges use unique integers. Dotted lines mark saved reference values.

**Keep your work:** expand **Save, load & export** and save variants to JSON before restarting. CSV export is available for current results. Edits remain drafts until saved. Changed inputs or selections clear old results; calculate again to refresh. Existing files require explicit overwrite confirmation.

Power: W · Time: h · Energy: Wh · Area: m². Preliminary sizing only.

## Baseline inputs
Edit the Python definitions as a fallback, including subsystem rows and safe overrides. Fractions here use 0–1; capacity retention `None` means TBD. Running all cells resets in-memory variants; save JSON before restarting.

Dependencies: `numpy`, `pandas`, `matplotlib`, `ipywidgets`. If needed, run `%pip install numpy pandas matplotlib ipywidgets` in a separate cell and restart the kernel. Without widgets, use `calculate(baseline, "nominal")` or `calculate(changed_case(baseline, {...}), "safe")` after running the calculation cell.

Code sections are collapsed to keep the notebook compact. Expand a code cell to edit its Python definitions.

In [1]:
from copy import deepcopy
from pathlib import Path
import json
import html
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "font.size": 10})


In [2]:
# PARAMETERS
mission = {
    "lifetime_years": 5.0,
    "days_per_year": 365.0,          # For the approximate cycle count
    "eclipse_h": 1.0,               # Update when orbit is confirmed
    "sunlight_h": 1.0,              # Assumed; must be > 0 for solar sizing
    "load_margin": 0.30,
}
solar_array = {
    "solar_irradiance_W_m2": 1361.0,
    "cell_efficiency": 0.30,
    "X_e": 0.60,
    "X_d": 0.80,
    "inherent_factor": 0.72,
    "annual_degradation": 0.005,
    "incidence_deg": 0.0,
}
battery = {
    "battery_to_load_efficiency": 0.90,
    "depth_of_discharge": 0.30,
    "battery_units": 1,             # N equal units sharing the required energy
    "battery_capacity_retention": None,  # EOL/temperature usable capacity / BOL nameplate
}
parameters = {**mission, **solar_array, **battery}

# LOADS                  active_W idle_W peak_W duty_e duty_d enabled
load_rows = [
    ("payload",    "Payload",                  5.6, 0.0, 5.6,  1.0, 1.0, True),
    ("mechanisms", "Structure and Mechanisms", 0.0, 0.0, 0.0,  0.0, 0.0, True),
    ("thermal",    "Thermal control",         15.0, 0.0, 15.0, 1.0, 0.0, True),
    ("eps",        "EPS electronics",          2.0, 0.0, 2.0,  1.0, 1.0, True),
    ("dsn",        "Communication DSN",       35.0, 0.0, 35.0, 0.1, 0.1, True),
    ("crosslink",  "Communication node-node", 11.0, 1.2, 11.0, 0.1, 0.1, True),
    ("broadcast",  "One-way surface broadcast",11.0,0.0,11.0, 1.0, 1.0, True),
    ("obc",        "OBC",                      2.0, 0.0, 4.0,  1.0, 1.0, True),
    ("adcs",       "ADCS",                     4.27,0.0, 8.12, 1.0, 1.0, True),
]
loads = pd.DataFrame(load_rows, columns=["id", "Subsystem", "active_W", "idle_W",
                                       "peak_W", "duty_e", "duty_d", "enabled"]).set_index("id")

# SAFE MODE: only list values that differ from nominal
safe_overrides = {
    "payload": {"enabled": False},
    "crosslink": {"enabled": False},
    "broadcast": {"enabled": False},
}


baseline = {"parameters": deepcopy(parameters),
            "loads": loads.to_dict(orient="index"),
            "safe_overrides": deepcopy(safe_overrides)}


## Calculation functions
Original equations and nominal/safe-mode logic. `calculate()` returns a subsystem table and a result Series. `changed_case()` validates an independent modified copy. BOL battery capacity remains TBD until retention is known.

In [3]:
def validate_case(case):
    """Reject invalid inputs before calculating; no silent clipping."""
    p = case["parameters"]
    for key, value in p.items():
        if value is None and key == "battery_capacity_retention":
            continue
        if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.number)) or not math.isfinite(value):
            raise ValueError(f"{key}: enter a finite number (retention may be None).")
    for key in ["lifetime_years", "eclipse_h", "load_margin"]:
        if p[key] < 0:
            raise ValueError(f"{key} must be >= 0.")
    for key in ["sunlight_h", "days_per_year", "solar_irradiance_W_m2"]:
        if p[key] <= 0:
            raise ValueError(f"{key} must be > 0.")
    for key in ["cell_efficiency", "X_e", "X_d", "inherent_factor",
                "battery_to_load_efficiency", "depth_of_discharge", "battery_capacity_retention"]:
        if p[key] is not None and not 0 < p[key] <= 1:
            raise ValueError(f"{key} must be in (0, 1].")
    if not 0 <= p["annual_degradation"] < 1:
        raise ValueError("annual_degradation must be in [0, 1).")
    if not 0 <= p["incidence_deg"] < 90:
        raise ValueError("incidence_deg must be in [0, 90); 90 degrees has no solar output.")
    if p["battery_units"] < 1 or int(p["battery_units"]) != p["battery_units"]:
        raise ValueError("battery_units must be a positive integer.")
    for key, override in case["safe_overrides"].items():
        if key not in case["loads"]:
            raise ValueError(f"Unknown safe-mode subsystem: {key}")
        if set(override) - set(case["loads"][key]):
            raise ValueError(f"Unknown safe-mode field for {key}.")
    for mode in ["nominal", "safe"]:
        for key, row in mode_loads(case, mode).items():
            if not isinstance(row["enabled"], (bool, np.bool_)):
                raise ValueError(f"{mode}.{key}.enabled must be True or False.")
            for field in ["active_W", "idle_W", "peak_W", "duty_e", "duty_d"]:
                v = row[field]
                if isinstance(v, (bool, np.bool_)) or not isinstance(v, (int, float, np.number)) or not math.isfinite(v) or v < 0:
                    raise ValueError(f"{mode}.{key}.{field} must be a finite non-negative number.")
            if max(row["duty_e"], row["duty_d"]) > 1:
                raise ValueError(f"{mode}.{key}: duty cycles must be in [0, 1].")
            if row["peak_W"] < max(row["active_W"], row["idle_W"]):
                raise ValueError(
                    f"{mode.capitalize()} · {row['Subsystem']}: Active = {row['active_W']:g} W, "
                    f"Idle = {row['idle_W']:g} W, Peak = {row['peak_W']:g} W. "
                    f"Set Peak to at least {max(row['active_W'], row['idle_W']):g} W, "
                    "or reduce Active/Idle to match the intended load. "
                    "For an active-power sweep, explicitly select the peak-power link if intended.")


def mode_loads(case, mode):
    rows = deepcopy(case["loads"])
    if mode == "safe":
        for key, override in case["safe_overrides"].items():
            rows[key].update(override)
    elif mode != "nominal":
        raise ValueError("mode must be 'nominal' or 'safe'.")
    return rows


def calculate(case, mode="nominal"):
    """Return a subsystem table and scalar results; never modify the inputs."""
    validate_case(case)
    p = case["parameters"]
    te, td = p["eclipse_h"], p["sunlight_h"]
    table = pd.DataFrame.from_dict(mode_loads(case, mode), orient="index")
    on = table["enabled"].astype(float)
    table["Eclipse [W]"] = on * (table.duty_e * table.active_W + (1-table.duty_e) * table.idle_W)
    table["Sunlight [W]"] = on * (table.duty_d * table.active_W + (1-table.duty_d) * table.idle_W)
    table["Orbit average [W]"] = (table["Eclipse [W]"] * te + table["Sunlight [W]"] * td) / (te+td)
    table["Peak [W]"] = on * table.peak_W
    pe, pd_, avg, peak = table[["Eclipse [W]", "Sunlight [W]", "Orbit average [W]", "Peak [W]"]].sum()
    factor = 1 + p["load_margin"]
    pe_req, pd_req, avg_req, peak_req = factor * np.array([pe, pd_, avg, peak])
    psa = (pe_req * te / p["X_e"] + pd_req * td / p["X_d"]) / td
    p0 = p["solar_irradiance_W_m2"] * p["cell_efficiency"]
    pbol = p0 * p["inherent_factor"] * math.cos(math.radians(p["incidence_deg"]))
    ld = (1 - p["annual_degradation"]) ** p["lifetime_years"]
    peol = pbol * ld
    eclipse_energy = pe_req * te
    capacity_bank = eclipse_energy / (p["depth_of_discharge"] * p["battery_to_load_efficiency"])
    retention = p["battery_capacity_retention"]
    results = {
        "Eclipse fraction [-]": te / (te+td),
        "Orbit period [h]": te+td,
        "Approximate cycles [-]": p["lifetime_years"] * p["days_per_year"] * 24 / (te+td) if te > 0 else 0,
        "Eclipse load [W]": pe, "Sunlight load [W]": pd_,
        "Orbit average [W]": avg, "Simultaneous peak [W]": peak,
        "Eclipse load + margin [W]": pe_req, "Sunlight load + margin [W]": pd_req,
        "Orbit average + margin [W]": avg_req, "Peak + margin [W]": peak_req,
        "Array power required at EOL [W]": psa,
        "Ideal cell output [W/m2]": p0, "BOL output [W/m2]": pbol,
        "Lifetime factor [-]": ld, "EOL output [W/m2]": peol,
        "Array area [m2]": psa / peol,
        "Eclipse energy to loads [Wh]": eclipse_energy,
        "Battery bank EOL [Wh]": capacity_bank,
        "Battery per unit EOL [Wh]": capacity_bank / p["battery_units"],
        "Battery bank BOL [Wh]": np.nan if retention is None else capacity_bank / retention,
        "Battery peak at terminals [W]": peak_req / p["battery_to_load_efficiency"],
    }
    return table, pd.Series(results, dtype=float)


def changed_case(case, changes):
    """Copy a case and apply dotted input paths; baseline stays unchanged.

    Examples: parameters.eclipse_h, loads.dsn.duty_e,
    safe_overrides.dsn.duty_e. Safe overrides can be added here.
    """
    new = deepcopy(case)
    for path, value in changes.items():
        parts = path.split(".")
        if len(parts) == 2 and parts[0] == "parameters" and parts[1] in new["parameters"]:
            new["parameters"][parts[1]] = value
        elif (len(parts) == 3 and parts[0] in ["loads", "safe_overrides"]
              and parts[1] in new["loads"] and parts[2] in new["loads"][parts[1]]):
            new[parts[0]].setdefault(parts[1], {})[parts[2]] = value
        else:
            raise KeyError(f"Unknown input path: {path}")
    validate_case(new)
    return new


_model_validate_case = validate_case


## Interface code
Metadata, validation, snapshot management and one widget controller. Tables and charts are explicit HTML/Image widget content so callback results do not depend on notebook output capture. All file writes and calculations require an explicit action.

In [4]:
# Metadata drives editor fields, conversion, sweep selectors, and schema checks.
# Limits use internal units; percentages are shown as 0–100 in the interface.
PARAMS = {
    'Mission': {
        'lifetime_years': ('Lifetime', 'years', float, False, 0, None),
        'days_per_year': ('Days per year', 'days', float, False, 0, None),
        'eclipse_h': ('Eclipse duration', 'h', float, False, 0, None),
        'sunlight_h': ('Sunlight duration', 'h', float, False, 0, None),
        'load_margin': ('Load margin', '%', float, True, 0, None),
    },
    'Solar array': {
        'solar_irradiance_W_m2': ('Solar irradiance', 'W/m²', float, False, 0, None),
        'cell_efficiency': ('Cell efficiency', '%', float, True, 0, 1),
        'X_e': ('Eclipse path efficiency', '%', float, True, 0, 1),
        'X_d': ('Sunlight path efficiency', '%', float, True, 0, 1),
        'inherent_factor': ('Inherent factor', '%', float, True, 0, 1),
        'annual_degradation': ('Annual degradation', '%/year', float, True, 0, 1),
        'incidence_deg': ('Incidence angle', 'deg', float, False, 0, 90),
    },
    'Battery': {
        'battery_to_load_efficiency': ('Battery-to-load efficiency', '%', float, True, 0, 1),
        'depth_of_discharge': ('Depth of discharge (DOD)', '%', float, True, 0, 1),
        'battery_units': ('Battery units', 'count', int, False, 1, None),
        'battery_capacity_retention': ('Capacity retention', '%', float, True, 0, 1),
    },
}
LOAD_META = {
    'active_W': ('Active', 'W', float, False, 0, None),
    'idle_W': ('Idle', 'W', float, False, 0, None),
    'peak_W': ('Peak', 'W', float, False, 0, None),
    'duty_e': ('Eclipse duty', '%', float, True, 0, 1),
    'duty_d': ('Sunlight duty', '%', float, True, 0, 1),
    'enabled': ('Enabled', '', bool, False, None, None),
}
PARAM_META = {k: v for group in PARAMS.values() for k, v in group.items()}
KEY_RESULTS = ['Orbit average + margin [W]', 'Peak + margin [W]',
               'Array power required at EOL [W]', 'Array area [m2]',
               'Battery bank EOL [Wh]', 'Battery bank BOL [Wh]']
OUTPUTS = list(calculate(baseline)[1].index)
FORMAT_VERSION = 1

# Keep the original model validation, adding structural checks for JSON input.

def validate_case(case):
    if not isinstance(case, dict) or set(case) != {'parameters', 'loads', 'safe_overrides'}:
        raise ValueError('A case needs parameters, loads and safe_overrides.')
    if not isinstance(case['parameters'], dict) or set(case['parameters']) != set(PARAM_META):
        raise ValueError('Parameter fields do not match this notebook.')
    if not isinstance(case['loads'], dict) or not case['loads']:
        raise ValueError('At least one subsystem is required.')
    fields = set(LOAD_META) | {'Subsystem'}
    for key, row in case['loads'].items():
        if not isinstance(key, str) or not key or '.' in key:
            raise ValueError('Subsystem IDs must be nonempty strings without dots.')
        if not isinstance(row, dict) or set(row) != fields or not isinstance(row['Subsystem'], str):
            raise ValueError(f'{key}: invalid subsystem fields.')
    if not isinstance(case['safe_overrides'], dict):
        raise ValueError('Safe overrides must be a dictionary.')
    for key, row in case['safe_overrides'].items():
        if not isinstance(row, dict) or set(row) - set(LOAD_META):
            raise ValueError(f'{key}: invalid safe override fields.')
    _model_validate_case(case)
    for key, meta in PARAM_META.items():
        value = case['parameters'][key]
        if value is not None and ((meta[4] is not None and value < meta[4]) or
                                  (meta[5] is not None and value > meta[5])):
            raise ValueError(f'{meta[0]} is outside its allowed range.')


def friendly_error(error):
    message = str(error)
    for key, meta in {**PARAM_META, **LOAD_META}.items():
        if key in message and meta[3]:
            message = message.replace('(0, 1]', '(0, 100]%').replace('[0, 1)', '[0, 100)%').replace('[0, 1]', '[0, 100]%')
        message = message.replace(key, meta[0])
    return message


def display_value(value, meta):
    return value * (100 if meta[3] else 1)


def internal_value(value, meta):
    return value / 100 if meta[3] else value


def differences(case):
    records = []
    def add(group, field, value, original, meta):
        if value != original:
            def fmt(v):
                return 'TBD' if v is None else display_value(v, meta)
            records.append({'Group / subsystem': group, 'Input': meta[0],
                            'Unit': meta[1], 'Baseline': fmt(original), 'Variant': fmt(value)})
    for key, meta in PARAM_META.items():
        add('Parameters', key, case['parameters'][key], baseline['parameters'][key], meta)
    for mode in ['nominal', 'safe']:
        current, original = mode_loads(case, mode), mode_loads(baseline, mode)
        for key, row in current.items():
            if key not in original:
                records.append({'Group / subsystem': row['Subsystem'], 'Input': 'Added subsystem'})
                continue
            for field, meta in LOAD_META.items():
                add(f'{mode}: {row["Subsystem"]}', field, row[field], original[key][field], meta)
    # Explicit equal-valued overrides still matter to future nominal edits.
    for key, row in case['safe_overrides'].items():
        for field, value in row.items():
            if value == case['loads'][key][field] and field not in baseline['safe_overrides'].get(key, {}):
                records.append({'Group / subsystem': f'safe: {key}', 'Input': LOAD_META[field][0],
                                'Variant': 'Explicit override (equals nominal)'})
    removed = set(baseline['loads']) - set(case['loads'])
    for key in sorted(removed):
        records.append({'Group / subsystem': key, 'Input': 'Removed subsystem'})
    return pd.DataFrame(records)


def comparison_frame(variants, names, mode, outputs, delta='None'):
    if not names or not outputs:
        raise ValueError('Select at least one variant and output.')
    reference = checked_calculate(baseline, mode)[1]
    rows = []
    for name in names:
        values = checked_calculate(variants[name], mode)[1][list(outputs)]
        row = {'Variant': name, 'Mode': mode, **values.to_dict()}
        for output in outputs:
            if delta == 'Absolute':
                row['Δ ' + output] = values[output] - reference[output]
            elif delta == 'Percentage':
                ref = reference[output]
                row['Δ % ' + output] = (100 * (values[output] - ref) / ref
                                          if pd.notna(ref) and ref != 0 else np.nan)
        rows.append(row)
    return pd.DataFrame(rows)


def sweep_cases(variants, names, mode, path, values, links=()):
    """Validate every point from an independent saved snapshot before returning results."""
    if not names or not len(values):
        raise ValueError('Select variants and at least one sweep value.')
    parts = path.split('.')
    if parts[0] == 'parameters':
        if len(parts) != 2 or parts[1] not in PARAM_META:
            raise ValueError('Choose a numeric parameter.')
        meta = PARAM_META[parts[1]]
    else:
        if len(parts) != 3 or parts[0] not in ('loads', 'safe_overrides') or parts[2] not in LOAD_META:
            raise ValueError('Choose a numeric load input.')
        meta = LOAD_META[parts[2]]
    if meta[2] is bool:
        raise ValueError('Only numeric inputs can be swept.')
    allowed = set()
    if path in ('parameters.eclipse_h', 'parameters.sunlight_h'):
        allowed.add('orbit')
    if len(parts) == 3:
        if parts[2] == 'active_W':
            allowed.add('peak')
        if parts[1] in ('dsn', 'crosslink', 'broadcast') and parts[2] in ('duty_e', 'duty_d'):
            allowed.add('duty')
    if set(links) - allowed:
        raise ValueError('A selected link does not apply to this input.')
    rows, errors = [], []
    for name in names:
        source = variants[name]
        for value in values:
            try:
                if isinstance(value, (bool, np.bool_)) or not math.isfinite(value):
                    raise ValueError('Use finite numbers.')
                if meta[2] is int and (value < 1 or int(value) != value):
                    raise ValueError('Battery units require positive integer values; use an explicit list or an integer-spaced range.')
                actual = internal_value(value, meta)
                if meta[2] is int:
                    actual = int(actual)
                changes = {path: actual}
                if 'orbit' in links:
                    other = 'sunlight_h' if parts[1] == 'eclipse_h' else 'eclipse_h'
                    p = source['parameters']
                    changes['parameters.' + other] = p['eclipse_h'] + p['sunlight_h'] - actual
                if 'duty' in links:
                    other = 'duty_d' if parts[2] == 'duty_e' else 'duty_e'
                    changes['.'.join(parts[:2] + [other])] = actual
                if 'peak' in links:
                    changes['.'.join(parts[:2] + ['peak_W'])] = actual
                case = changed_case(source, changes)
                result = checked_calculate(case, mode)[1]
                rows.append({'Variant': name, 'Mode': mode, 'Input value': value, **result.to_dict()})
            except (ValueError, KeyError, TypeError, OverflowError) as error:
                errors.append(f'{name}, {value:g} {meta[1]}: {friendly_error(error)}')
    if errors:
        raise ValueError('Invalid sweep — no plot produced.\n' + '\n'.join(errors[:12]) +
                         (f'\n… {len(errors)-12} more invalid points.' if len(errors) > 12 else ''))
    return pd.DataFrame(rows)


def atomic_text(path, text):
    import os
    import tempfile
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', dir=Path(path).parent,
                                         prefix='.power-budget-', delete=False) as stream:
            temporary = Path(stream.name)
            stream.write(text)
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()


def write_variants(path, variants):
    for case in variants.values():
        for mode in ('nominal', 'safe'):
            checked_calculate(case, mode)
    data = {'format_version': FORMAT_VERSION,
            'variants': {k: deepcopy(v) for k, v in variants.items() if k != 'Baseline'}}
    atomic_text(path, json.dumps(data, indent=2, allow_nan=False))


def read_variants(path):
    with open(path, encoding='utf-8') as stream:
        data = json.load(stream)
    if not isinstance(data, dict) or type(data.get('format_version')) is not int or data['format_version'] != FORMAT_VERSION:
        raise ValueError('Unsupported variants format version.')
    items = data.get('variants')
    if not isinstance(items, dict):
        raise ValueError('File must contain a variants dictionary.')
    result = {}
    for name, case in items.items():
        if not isinstance(name, str) or not name.strip() or name != name.strip() or name == 'Baseline':
            raise ValueError('Loaded names must be nonempty; Baseline is protected.')
        for mode in ('nominal', 'safe'):
            checked_calculate(case, mode)
        result[name] = deepcopy(case)
    return result


try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

def fmt_number(value):
    return 'TBD' if pd.isna(value) else f'{value:,.6g}'


def table_widget(frame, index=False, unknown='TBD'):
    # Explicit widget content avoids output-capture differences between notebook clients.
    rendered = frame.map(lambda v: fmt_number(v) if isinstance(v, (float, np.floating)) else v) if hasattr(frame, 'map') else frame.applymap(lambda v: fmt_number(v) if isinstance(v, (float, np.floating)) else v)
    return widgets.HTML(rendered.fillna(unknown).to_html(index=index, escape=True, border=0),
                        layout=widgets.Layout(width='100%', overflow='auto'))


def check_values(box):
    return tuple(c.description for c in box.children if c.value)


def set_checks(box, options, selected, callback):
    controls = []
    for label in options:
        control = widgets.Checkbox(description=label, value=label in selected, indent=False,
                                   layout=widgets.Layout(width='auto', min_width='180px'))
        control.observe(callback, names='value')
        controls.append(control)
    box.children = tuple(controls)


def checked_cases(variants, names, mode):
    if not names:
        raise ValueError('Choose at least one saved variant.')
    return {name: checked_calculate(variants[name], mode) for name in names}


def checked_calculate(case, mode):
    with np.errstate(over='raise', divide='raise', invalid='raise'):
        try:
            table, results = calculate(case, mode)
        except (ArithmeticError, OverflowError) as error:
            raise ValueError('These inputs exceed the numerical range of this sizing model. Check durations, lifetime and efficiencies.') from error
    required = results.drop('Battery bank BOL [Wh]') if case['parameters']['battery_capacity_retention'] is None else results
    if not np.isfinite(required.to_numpy()).all():
        raise ValueError('The calculation produced a non-finite result. Check durations, lifetime and efficiencies.')
    return table, results


def comparison_display(frame, outputs, delta):
    """Metrics down the page, variants across; no truncated DataFrame columns."""
    rows = []
    for output in outputs:
        label, unit = output.rsplit(' [', 1)
        row = {'Result': label, 'Unit': unit.rstrip(']').replace('m2', 'm²')}
        for _, values in frame.iterrows():
            row[values['Variant']] = fmt_number(values[output])
        rows.append(row)
        prefix = 'Δ ' if delta == 'Absolute' else 'Δ % '
        if delta != 'None':
            row = {'Result': '↳ Change from baseline', 'Unit': unit.rstrip(']').replace('m2', 'm²') if delta == 'Absolute' else '%'}
            for _, values in frame.iterrows():
                value = values[prefix + output]
                row[values['Variant']] = '—' if pd.isna(value) else f'{value:+.6g}'
            rows.append(row)
    return pd.DataFrame(rows)


def area_explanation(results):
    rows = []
    for name, (_, result) in results.items():
        rows.append({'Variant': name,
                     'Required array power [W]': result['Array power required at EOL [W]'],
                     'EOL array output [W/m²]': result['EOL output [W/m2]'],
                     'Area = power ÷ output [m²]': result['Array area [m2]']})
    return pd.DataFrame(rows)


def result_charts(frame, outputs, mode, sweep=None):
    """Return PNG widgets from Matplotlib; works with inline and noninteractive backends."""
    import io
    charts = []
    names = tuple(dict.fromkeys(frame['Variant']))
    palette = plt.get_cmap('tab10')
    colors = {name: ('#64748b' if name == 'Baseline' else palette(i % 10)) for i, name in enumerate(names)}
    units = {}
    for output in outputs:
        units.setdefault(output.rsplit('[', 1)[-1].rstrip(']'), []).append(output)
    for unit, metrics in units.items():
        unit = unit.replace('m2', 'm²')
        cols = min(2, len(metrics))
        rows = math.ceil(len(metrics)/cols)
        fig, axes = plt.subplots(rows, cols, figsize=(9, 3.1*rows), squeeze=False)
        try:
            for ax, metric in zip(axes.flat, metrics):
                if sweep:
                    meta, references = sweep
                    for name in names:
                        points = frame[frame['Variant'] == name].sort_values('Input value')
                        ax.plot(points['Input value'], points[metric], marker='.', color=colors[name], label=name)
                        ref = references[name]
                        if ref is not None and points['Input value'].min() <= ref <= points['Input value'].max():
                            ax.axvline(ref, linestyle=':', linewidth=1, color=colors[name], alpha=.7)
                    ax.set_xlabel(f'{meta[0]} [{meta[1]}]')
                    ax.set_ylabel(unit)
                    ax.legend(fontsize=8)
                else:
                    positions = np.arange(len(frame))
                    vals = frame[metric].to_numpy()
                    ax.barh(positions, vals, color=[colors[n] for n in frame['Variant']])
                    ax.set_yticks(positions, labels=frame['Variant'])
                    ax.invert_yaxis()
                    ax.set_xlabel(unit)
                    for y, value in zip(positions, vals):
                        ax.annotate(fmt_number(value), (0 if pd.isna(value) else value, y),
                                    xytext=(4, 0), textcoords='offset points', va='center', fontsize=8)
                    ax.margins(x=.22)
                if frame[metric].isna().all():
                    ax.text(.5, .5, 'TBD · set battery retention', transform=ax.transAxes, ha='center', fontsize=9)
                ax.set_title(metric.split(' [')[0], loc='left', fontsize=10)
                ax.spines[['top', 'right']].set_visible(False)
            for ax in list(axes.flat)[len(metrics):]:
                ax.set_visible(False)
            fig.suptitle(f'{mode.capitalize()} · {unit}', x=.02, ha='left', fontsize=11)
            fig.tight_layout()
            buffer = io.BytesIO()
            fig.savefig(buffer, format='png', dpi=125, bbox_inches='tight')
            charts.append(widgets.Image(value=buffer.getvalue(), format='png',
                                        layout=widgets.Layout(width='100%', max_width='1000px')))
        finally:
            plt.close(fig)
    return charts


class PowerBudgetUI:
    """Notebook controller with explicit draft, saved snapshot and result states."""
    def __init__(self):
        self.variants = {'Baseline': deepcopy(baseline)}
        self.current, self.draft = 'Baseline', deepcopy(baseline)
        self.busy, self.ready = False, False
        self.comparison_data = self.sweep_data = None
        self.comparison_fresh = self.sweep_fresh = False
        self.pending_write = self.pending_delete = None
        self.disk_dirty = False
        self.status = widgets.HTML()
        self.tab_status = [widgets.HTML() for _ in range(3)]
        self.selector = widgets.Dropdown(options=['Baseline'], description='Editing:', layout=widgets.Layout(width='300px'))
        self.name = widgets.Text(description='Copy name:', placeholder='Optional — a name is suggested')
        self.dirty_label, self.validation, self.orbit_summary = widgets.HTML(), widgets.HTML(), widgets.HTML()
        self.mode = widgets.ToggleButtons(options=[('Nominal loads', 'nominal'), ('Safe-mode loads', 'safe')])
        self.mission_style = widgets.Dropdown(options=['Durations', 'Period + eclipse fraction'], description='Define orbit by:', style={'description_width': 'initial'}, layout=widgets.Layout(width='350px'))
        self.form, self.diff_out, self.editor_result = widgets.VBox(), widgets.VBox(), widgets.VBox()
        self.save_button = self.button('Save changes', self.save, area=0)
        self.save_compare_button = self.button('Save & compare', self.save_and_compare, area=0, primary=True)
        self.discard_button = self.button('Discard edits', self.discard, area=0)
        self.delete_button = self.button('Delete variant', self.delete, area=0)
        edit = widgets.VBox([
            self.note('1 · Create a copy', 'Baseline is protected. Create a variant, edit its inputs, then Save & compare.'),
            self.row(self.selector, self.dirty_label),
            self.row(self.name, self.button('Create variant', self.duplicate, area=0)),
            self.accordion(self.row(self.delete_button, self.button('Cancel deletion', self.cancel_delete, area=0)), 'Manage variant'),
            self.note('2 · Edit inputs', 'Enter 30 for 30%. Hover over inputs for labels and units.'),
            self.row(self.mission_style, self.mode), self.orbit_summary, self.form,
            self.accordion(self.diff_out, 'Changes from baseline'), self.validation,
            self.row(self.save_compare_button, self.save_button, self.discard_button,
                     self.button('Preview draft', self.calculate_draft, area=0)),
            self.tab_status[0], self.editor_result])
        compare, sweep = self.build_compare(), self.build_sweep()
        self.tabs = widgets.Tab(children=[edit, compare, sweep])
        for i, title in enumerate(['Edit variants', 'Compare', 'Sweep']):
            self.tabs.set_title(i, title)
        self.file_name = widgets.Text(value='power_variants.json', description='JSON file:')
        self.csv_name = widgets.Text(value='power_comparison.csv', description='CSV file:')
        self.csv_source = widgets.Dropdown(options=['Comparison', 'Sweep'], description='Export:')
        self.csv_source.observe(lambda _: self.update_export(), names='value')
        self.overwrite = widgets.Checkbox(value=False, description='Confirm overwrite', indent=False)
        self.export_button = self.button('Export CSV', self.export_csv)
        files = widgets.VBox([
            widgets.HTML('JSON preserves complete variants. Baseline comes from the Python inputs. Files are written in the notebook working directory.'),
            self.row(self.file_name, self.button('Save variants JSON', self.save_json), self.button('Load variants JSON', self.load_json)),
            self.row(self.csv_source, self.csv_name, self.export_button), self.overwrite,
            widgets.HTML('<small>Only current results can be exported. Blank CSV values mean TBD or an undefined percentage.</small>')])
        style = widgets.HTML('''<style>
        .power-budget table {border-collapse:collapse; width:100%; font-size:13px;}
        .power-budget th, .power-budget td {padding:7px 10px; text-align:right; border-bottom:1px solid #cbd5e1; white-space:nowrap;}
        .power-budget th:first-child, .power-budget td:first-child {text-align:left;}
        .power-budget th {font-weight:600;}
        </style>''')
        self.root = widgets.VBox([style, widgets.HTML('<h2 style="margin:4px 0">Spacecraft power budget</h2><small>Preliminary sizing · independent variants · explicit calculations</small>'),
                                  self.status, self.tabs, self.accordion(files, 'Save, load & export')],
                                 layout=widgets.Layout(width='100%', max_width='1150px'))
        self.root.add_class('power-budget')
        self.selector.observe(self.switch, names='value')
        self.mode.observe(lambda _: self.render_editor(), names='value')
        self.mission_style.observe(lambda _: self.render_editor(), names='value')
        self.ready = True
        self.render_editor()
        self.update_export()
        self.message('Create a variant to begin. Baseline is available in Compare.')

    @staticmethod
    def row(*children):
        return widgets.HBox(children, layout=widgets.Layout(flex_flow='row wrap', align_items='center', gap='8px'))

    @staticmethod
    def note(title, text):
        return widgets.HTML(f'<b>{title}</b><br><small>{text}</small>')

    @staticmethod
    def accordion(child, title):
        box = widgets.Accordion(children=[child], selected_index=None)
        box.set_title(0, title)
        return box

    def button(self, label, callback, area=None, primary=False):
        button = widgets.Button(description=label, button_style='primary' if primary else '', layout=widgets.Layout(width='auto'))
        def guarded(_):
            button.disabled = True
            try:
                callback()
            except (ValueError, KeyError, TypeError, OSError, ArithmeticError) as error:
                self.message(friendly_error(error), error=True, area=area)
            finally:
                button.disabled = False
                if self.ready:
                    self.update_dirty()
                    self.update_export()
        button.on_click(guarded)
        return button

    def message(self, message, error=False, area=None):
        value = f'<div role="status" style="padding:8px;border-left:3px solid {"#b91c1c" if error else "#2563eb"};white-space:pre-wrap">{html.escape(str(message))}</div>'
        self.status.value = value
        if area is not None:
            self.tab_status[area].value = value if error else ''

    @property
    def dirty(self):
        return self.draft != self.variants[self.current]

    def update_dirty(self):
        self.dirty_label.value = '<b>● Unsaved draft</b>' if self.dirty else ('Protected baseline' if self.current == 'Baseline' else '✓ Saved in session')
        self.save_button.disabled = self.current == 'Baseline' or not self.dirty
        self.save_compare_button.disabled = self.current == 'Baseline'
        self.discard_button.disabled = not self.dirty
        self.delete_button.disabled = self.current == 'Baseline'

    def invalidate(self, which=None):
        if not self.ready:
            return
        for kind in ([which] if which else ['comparison', 'sweep']):
            setattr(self, kind + '_fresh', False)
            out = self.compare_out if kind == 'comparison' else self.sweep_out
            # Old results are removed so they cannot be mistaken for current calculations.
            out.children = (widgets.HTML('<i>Inputs or selections changed. ' + ('Update comparison' if kind == 'comparison' else 'Run sweep') + ' to see current results.</i>'),)
        self.status.value = '<small>' + ('● Unsaved draft. Use Save & compare to include your edits.' if self.dirty else 'Results need updating after the latest changes.') + '</small>'
        self.update_export()

    def require_clean(self):
        if self.dirty:
            raise ValueError(f'“{self.current}” has unsaved edits. Save changes or discard edits before continuing.')

    def require_analysis_ready(self):
        if self.dirty:
            raise ValueError(f'“{self.current}” has unsaved edits. Use Save & compare in Edit variants, or discard edits. Analysis uses saved variants.')

    def refresh_names(self, select_new=()):
        self.busy = True
        self.selector.options = list(self.variants)
        self.selector.value = self.current
        for box, callback in [(self.compare_names, lambda _: self.invalidate('comparison')),
                              (self.sweep_names, lambda _: self.sweep_selection_changed())]:
            selected = [n for n in check_values(box) if n in self.variants]
            if select_new:
                selected = list(dict.fromkeys(['Baseline', *select_new]))
            set_checks(box, self.variants, selected or ['Baseline'], callback)
        self.busy = False
        self.update_sweep_choices()
        self.invalidate()

    def switch(self, change):
        if self.busy:
            return
        if self.dirty:
            self.busy = True
            self.selector.value = self.current
            self.busy = False
            self.message('Save changes or discard edits before switching variants.', True, area=0)
            return
        self.pending_delete = None
        self.current = change['new']
        self.draft = deepcopy(self.variants[self.current])
        self.render_editor()
        self.message(f'Editing {self.current}.', area=0)

    def duplicate(self):
        self.require_clean()
        name = self.name.value.strip()
        if not name:
            i = 1
            while f'Variant {i}' in self.variants:
                i += 1
            name = f'Variant {i}'
        if name in self.variants or len(name) > 80:
            raise ValueError('Choose a unique variant name of at most 80 characters.')
        self.variants[name] = deepcopy(self.variants[self.current])
        self.current, self.draft = name, deepcopy(self.variants[name])
        self.name.value = ''
        self.disk_dirty = True
        self.refresh_names(select_new=[name])
        self.render_editor()
        self.message(f'Created {name}. Edit inputs below, then Save & compare.', area=0)

    def save(self):
        if self.current == 'Baseline':
            raise ValueError('Baseline is protected. Create a variant first.')
        validate_case(self.draft)
        # Ensure every saved case can actually calculate, including extreme finite inputs.
        for mode in ('nominal', 'safe'):
            checked_calculate(self.draft, mode)
        self.variants[self.current] = deepcopy(self.draft)
        self.disk_dirty = True
        self.update_dirty()
        self.invalidate()
        self.message(f'Saved {self.current} in this session. Save variants JSON to keep it after restart.', area=0)

    def save_and_compare(self):
        self.save()
        set_checks(self.compare_names, self.variants, ['Baseline', self.current], lambda _: self.invalidate('comparison'))
        self.compare_mode.value = self.mode.value
        self.tabs.selected_index = 1
        self.update_comparison()

    def discard(self):
        self.draft = deepcopy(self.variants[self.current])
        self.render_editor()
        self.message('Unsaved edits discarded.', area=0)

    def cancel_delete(self):
        self.pending_delete = None
        self.message('Deletion cancelled.', area=0)

    def delete(self):
        self.require_clean()
        if self.current == 'Baseline':
            raise ValueError('Baseline cannot be deleted.')
        if self.pending_delete != self.current:
            self.pending_delete = self.current
            self.message(f'Delete “{self.current}”? Click Delete variant again to confirm, or Cancel deletion.', area=0)
            return
        del self.variants[self.current]
        self.pending_delete = None
        self.current, self.draft = 'Baseline', deepcopy(baseline)
        self.disk_dirty = True
        self.refresh_names()
        self.render_editor()
        self.message('Variant deleted from this session.', area=0)


    def number(self, meta, value, description=True):
        cls = widgets.IntText if meta[2] is int else widgets.FloatText
        return cls(value=display_value(value, meta), tooltip=f'{meta[0]} [{meta[1]}]' + (' — enter 30 for 30%' if meta[3] else ''),
                   description=f'{meta[0]} [{meta[1]}]' if description else '',
                   style={'description_width': 'initial'},
                   layout=widgets.Layout(width='320px' if description else '98px'))

    def render_editor(self):
        self.busy = True
        self.editor_result.children = ()
        self.parameter_controls = {}
        sections = []
        p = self.draft['parameters']
        for group, fields in PARAMS.items():
            children = []
            if group == 'Mission':
                if self.mission_style.value == 'Period + eclipse fraction':
                    total = p['eclipse_h'] + p['sunlight_h']
                    self.period = self.number(('Orbit period', 'h', float, False, 0, None), total)
                    self.fraction = self.number(('Eclipse fraction', '%', float, True, 0, 1), p['eclipse_h']/total if total else 0)
                    children += [self.period, self.fraction]
                    self.period.observe(self.edit_changed, names='value')
                    self.fraction.observe(self.edit_changed, names='value')
            for key, meta in fields.items():
                if group == 'Mission' and key in ('eclipse_h', 'sunlight_h') and self.mission_style.value != 'Durations':
                    continue
                control = self.number(meta, p[key] if p[key] is not None else 1.0)
                self.parameter_controls[key] = control
                control.observe(self.edit_changed, names='value')
                if key == 'battery_capacity_retention':
                    self.retention_tbd = widgets.Checkbox(value=p[key] is None, description='TBD', indent=False, layout=widgets.Layout(width='80px'))
                    self.retention_tbd.observe(self.edit_changed, names='value')
                    children.append(widgets.HBox([control, self.retention_tbd]))
                else:
                    children.append(control)
            help_text = {
                'Mission': 'Margin multiplies loads by (1 + margin). Orbit input choices derive the same two durations.',
                'Solar array': 'Efficiencies and inherent factor: >0–100%. Annual degradation: 0–<100%; incidence: 0–<90°.',
                'Battery': 'DOD: usable fraction per cycle. Retention: EOL/temperature usable capacity ÷ BOL nameplate (>0–100%). TBD leaves BOL sizing unknown.',
            }[group]
            children.append(widgets.HTML(f'<small>{help_text}</small>'))
            sections.append(widgets.VBox([widgets.HTML(f'<b>{group}</b>'), *children], layout=widgets.Layout(width='330px', max_width='100%')))
        self.load_controls, self.override_controls = {}, {}
        rows = [widgets.HTML('<b>Loads</b><br><small>Safe mode: checked override boxes enable editing; uncheck to inherit nominal. Grey fields inherit nominal. Power ≥0; peak must cover active and idle. Duty: 0–100%.</small>')]
        header = [widgets.HTML('Subsystem', layout=widgets.Layout(width='185px'))]
        for meta in LOAD_META.values():
            header.append(widgets.HTML(f'{meta[0]} {"["+meta[1]+"]" if meta[1] else ""}', layout=widgets.Layout(width='98px')))
        rows.append(widgets.HBox(header))
        effective = mode_loads(self.draft, self.mode.value)
        for key, row in effective.items():
            columns = [widgets.HTML(html.escape(row['Subsystem']), layout=widgets.Layout(width='185px'))]
            for field, meta in LOAD_META.items():
                control = (widgets.Checkbox(value=row[field], indent=False, layout=widgets.Layout(width='98px'))
                           if meta[2] is bool else self.number(meta, row[field], False))
                self.load_controls[key, field] = control
                control.observe(self.edit_changed, names='value')
                if self.mode.value == 'safe':
                    override = widgets.Checkbox(value=field in self.draft['safe_overrides'].get(key, {}),
                                                description='Override', indent=False, layout=widgets.Layout(width='98px'))
                    self.override_controls[key, field] = override
                    override.observe(self.override_changed, names='value')
                    columns.append(widgets.VBox([control, override]))
                else:
                    columns.append(control)
            rows.append(widgets.HBox(columns))
        load_box = widgets.VBox(rows, layout=widgets.Layout(overflow='auto'))
        self.form.children = (widgets.HBox(sections, layout=widgets.Layout(flex_flow='row wrap', gap='12px', align_items='flex-start')), load_box)
        self.busy = False
        self.apply_disabled()
        self.update_dirty()
        self.show_diff()
        self.validate_draft()

    def apply_disabled(self):
        protected = self.current == 'Baseline'
        for key, control in self.parameter_controls.items():
            control.disabled = protected or (key == 'battery_capacity_retention' and self.retention_tbd.value)
        self.retention_tbd.disabled = protected
        if self.mission_style.value != 'Durations':
            self.period.disabled = self.fraction.disabled = protected
        for pair, control in self.load_controls.items():
            control.disabled = protected or (self.mode.value == 'safe' and not self.override_controls[pair].value)
        for control in self.override_controls.values():
            control.disabled = protected

    def edit_changed(self, _):
        if self.busy or self.current == 'Baseline':
            return
        p = self.draft['parameters']
        for key, control in self.parameter_controls.items():
            p[key] = internal_value(control.value, PARAM_META[key])
        if self.retention_tbd.value:
            p['battery_capacity_retention'] = None
        if self.mission_style.value != 'Durations':
            p['eclipse_h'] = self.period.value * self.fraction.value / 100
            p['sunlight_h'] = self.period.value * (1 - self.fraction.value / 100)
        for (key, field), control in self.load_controls.items():
            if self.mode.value == 'nominal':
                self.draft['loads'][key][field] = internal_value(control.value, LOAD_META[field])
            elif self.override_controls[key, field].value:
                self.draft['safe_overrides'].setdefault(key, {})[field] = internal_value(control.value, LOAD_META[field])
        self.apply_disabled()
        self.update_dirty()
        self.show_diff()
        self.validate_draft()
        self.editor_result.children = ()
        self.invalidate()

    def override_changed(self, _):
        if self.busy or self.current == 'Baseline':
            return
        for (key, field), control in self.override_controls.items():
            if control.value:
                self.draft['safe_overrides'].setdefault(key, {})[field] = internal_value(self.load_controls[key, field].value, LOAD_META[field])
            else:
                self.draft['safe_overrides'].get(key, {}).pop(field, None)
                if not self.draft['safe_overrides'].get(key):
                    self.draft['safe_overrides'].pop(key, None)
        self.render_editor()
        self.invalidate()

    def show_diff(self):
        frame = differences(self.draft)
        self.diff_out.children = (table_widget(frame) if not frame.empty else widgets.HTML('No differences from baseline.'),)

    def validate_draft(self):
        p = self.draft['parameters']
        total = p['eclipse_h'] + p['sunlight_h']
        self.orbit_summary.value = f"<small>Derived orbit: eclipse {p['eclipse_h']:.6g} h · sunlight {p['sunlight_h']:.6g} h · period {total:.6g} h</small>"
        for control in self.load_controls.values():
            control.layout.border = ''
        try:
            validate_case(self.draft)
        except (ValueError, KeyError, TypeError) as error:
            self.validation.value = '<div role="alert" style="color:#b91c1c;white-space:pre-wrap">' + html.escape(friendly_error(error)) + '</div>'
            for key, row in mode_loads(self.draft, self.mode.value).items():
                if row['peak_W'] < max(row['active_W'], row['idle_W']):
                    for field in ('active_W', 'idle_W', 'peak_W'):
                        self.load_controls[key, field].layout.border = '1px solid #b91c1c'
        else:
            self.validation.value = '<small>✓ Inputs valid in nominal and safe mode. Changes require Save.</small>'

    def calculate_draft(self):
        table, results = checked_calculate(self.draft, self.mode.value)
        frame = pd.DataFrame({'Result': KEY_RESULTS, 'Draft': results[KEY_RESULTS].to_numpy()})
        self.editor_result.children = (self.note('Draft preview', 'Preview only. Save & compare commits this draft and compares it with Baseline.'), table_widget(frame))
        self.message('Draft validated and calculated.', area=0)



    def output_picker(self, kind):
        box = widgets.VBox(layout=widgets.Layout(max_height='210px', overflow='auto'))
        set_checks(box, OUTPUTS, KEY_RESULTS, lambda _: self.invalidate(kind))
        def preset(selected):
            set_checks(box, OUTPUTS, selected, lambda _: self.invalidate(kind))
            self.invalidate(kind)
        panel = widgets.VBox([self.row(self.button('Key sizing results', lambda: preset(KEY_RESULTS)),
                                       self.button('All outputs', lambda: preset(OUTPUTS)),
                                       self.button('Clear', lambda: preset([]))), box])
        return box, self.accordion(panel, 'Choose outputs · six key results selected by default')

    def names_picker(self, kind):
        box = widgets.VBox(layout=widgets.Layout(max_height='150px', overflow='auto', min_width='220px'))
        callback = lambda _: self.invalidate(kind) if kind == 'comparison' else self.sweep_selection_changed()
        set_checks(box, self.variants, ['Baseline'], callback)
        return box

    def build_compare(self):
        self.compare_names = self.names_picker('comparison')
        self.compare_mode = widgets.ToggleButtons(options=[('Nominal', 'nominal'), ('Safe mode', 'safe')])
        self.compare_outputs, outputs_panel = self.output_picker('comparison')
        self.delta = widgets.Dropdown(options=[('Values only', 'None'), ('Absolute difference', 'Absolute'), ('Percentage difference', 'Percentage')], description='Baseline change:', style={'description_width': 'initial'}, layout=widgets.Layout(width='330px'))
        self.compare_out = widgets.VBox([widgets.HTML('<i>Choose variants, then Update comparison. Save & compare in the editor selects your variant automatically.</i>')])
        self.compare_out.add_class('power-comparison')
        for control in (self.compare_mode, self.delta):
            control.observe(lambda _: self.invalidate('comparison'), names='value')
        return widgets.VBox([self.note('Compare saved variants', 'Tick variants to compare. Save & compare in the editor saves your draft and selects it with Baseline.'),
                             self.compare_names, self.row(self.compare_mode, self.delta), outputs_panel,
                             self.row(self.button('Update comparison', self.update_comparison, area=1, primary=True),
                                      self.button('Back to editor', lambda: setattr(self.tabs, 'selected_index', 0), area=1)),
                             self.tab_status[1], self.compare_out])

    def update_comparison(self):
        self.comparison_fresh = False
        self.compare_out.children = ()
        self.update_export()
        self.require_analysis_ready()
        names, outputs, mode = check_values(self.compare_names), check_values(self.compare_outputs), self.compare_mode.value
        if not outputs:
            raise ValueError('Choose at least one output in Choose outputs.')
        results = checked_cases(self.variants, names, mode)
        frame = comparison_frame(self.variants, names, mode, outputs, self.delta.value)
        notes = []
        baseline_result = checked_calculate(baseline, mode)[1]
        for name in names:
            if name == 'Baseline':
                continue
            if self.variants[name] == baseline:
                notes.append(f'{name} is an unchanged copy of Baseline. Edit inputs, then Save & compare.')
            elif 'Array area [m2]' in outputs and results[name][1]['Array area [m2]'] == baseline_result['Array area [m2]']:
                notes.append(f'{name} has the same array area as Baseline. The power-to-output ratio is unchanged; see the explanation below.')
        children = [self.note(f'{mode.capitalize()} · {len(names)} saved variant(s)', 'TBD = unknown retention. — = undefined difference (zero or unknown baseline). Numbers use six significant digits.')]
        if notes:
            children.append(widgets.HTML('<div style="padding:8px;border-left:3px solid #d97706">' + '<br>'.join(html.escape(n) for n in notes) + '</div>'))
        children.append(table_widget(comparison_display(frame, outputs, self.delta.value)))
        if 'Array area [m2]' in outputs:
            children += [self.note('Why these array areas?', 'Area = required EOL array power ÷ EOL output per m². Average loads drive array power. A peak-only or battery-only change does not change array area.'),
                         table_widget(area_explanation(results))]
        children.extend(result_charts(frame, outputs, mode))
        details = []
        for name in names:
            details += [widgets.HTML(f'<b>{html.escape(name)} · {mode}</b>'),
                        table_widget(results[name][0][['Subsystem', 'Eclipse [W]', 'Sunlight [W]', 'Orbit average [W]', 'Peak [W]']])]
        children.append(self.accordion(widgets.VBox(details), 'Detailed subsystem loads'))
        self.compare_out.children = tuple(children)
        self.comparison_data, self.comparison_fresh = frame.copy(), True
        self.update_export()
        self.message('Comparison updated from the selected saved variants.', area=1)


    def build_sweep(self):
        self.sweep_names = self.names_picker('sweep')
        self.sweep_mode = widgets.ToggleButtons(options=[('Nominal', 'nominal'), ('Safe mode', 'safe')])
        self.sweep_outputs, outputs_panel = self.output_picker('sweep')
        self.group = widgets.Dropdown(options=list(PARAMS) + ['Loads'], description='Group:')
        self.subsystem = widgets.Dropdown(description='Subsystem:')
        self.variable = widgets.Dropdown(description='Variable:', style={'description_width': 'initial'}, layout=widgets.Layout(width='380px'))
        self.range_style = widgets.ToggleButtons(options=['Range', 'Explicit list'])
        self.minimum = widgets.FloatText(value=.5, description='Minimum:', layout=widgets.Layout(width='220px'))
        self.maximum = widgets.FloatText(value=1.5, description='Maximum:', layout=widgets.Layout(width='220px'))
        self.points = widgets.IntText(value=11, description='Points:', layout=widgets.Layout(width='180px'))
        self.range_row = self.row(self.minimum, self.maximum, self.points)
        self.explicit = widgets.Text(value='0.5, 1, 1.5', description='Values:', placeholder='Comma-separated numbers', layout=widgets.Layout(width='480px'))
        self.sweep_hint, self.links_label = widgets.HTML(), widgets.HTML()
        self.link_orbit = widgets.Checkbox(description='Hold each variant’s orbit period constant', indent=False, layout=widgets.Layout(width='auto'))
        self.link_duty = widgets.Checkbox(description='Keep communication eclipse and sunlight duty equal', indent=False, layout=widgets.Layout(width='auto'))
        self.link_peak = widgets.Checkbox(description='Set peak power equal to active power', indent=False, layout=widgets.Layout(width='auto'))
        self.sweep_out = widgets.VBox()
        self.sweep_out.add_class('power-sweep')
        self.group.observe(lambda _: self.update_sweep_choices(), names='value')
        for control in (self.subsystem, self.variable, self.sweep_mode):
            control.observe(lambda _: self.update_sweep_parameter(), names='value')
        for control in (self.link_orbit, self.link_duty, self.link_peak):
            control.observe(lambda _: self.links_changed(), names='value')
        for control in (self.minimum, self.maximum, self.points, self.explicit):
            control.observe(lambda _: self.invalidate('sweep') if not self.busy else None, names='value')
        self.range_style.observe(lambda _: self.update_range_style(), names='value')
        self.update_sweep_choices()
        self.variable.value = 'eclipse_h'
        self.update_range_style()
        return widgets.VBox([self.note('1 · Select variants and input', 'Every sweep point starts from its saved variant. Safe-mode load sweeps create temporary safe overrides.'),
                             self.sweep_names, self.sweep_mode, self.row(self.group, self.subsystem), self.variable, self.sweep_hint,
                             self.note('2 · Set values and optional links', 'Links reset when the input, mode or selected variants change.'),
                             self.range_style, self.range_row, self.explicit,
                             self.button('Suggest range from saved value', self.suggest_range, area=2),
                             self.link_orbit, self.link_duty, self.link_peak, self.links_label, outputs_panel,
                             self.button('Run sweep', self.run_sweep, area=2, primary=True), self.tab_status[2], self.sweep_out])

    def sweep_selection_changed(self):
        if not self.busy:
            self.update_sweep_choices()
            self.invalidate('sweep')

    def update_range_style(self):
        explicit = self.range_style.value == 'Explicit list'
        self.explicit.layout.display = '' if explicit else 'none'
        self.range_row.layout.display = 'none' if explicit else 'flex'
        self.invalidate('sweep')

    def update_sweep_choices(self):
        if self.busy:
            return
        self.busy = True
        old, old_subsystem = self.variable.value, self.subsystem.value
        if self.group.value == 'Loads':
            cases = [self.variants[name] for name in check_values(self.sweep_names)]
            keys = set.intersection(*(set(c['loads']) for c in cases)) if cases else set(baseline['loads'])
            source = cases[0] if cases else baseline
            self.subsystem.options = [(source['loads'][k]['Subsystem'], k) for k in source['loads'] if k in keys]
            self.subsystem.value = old_subsystem if old_subsystem in keys else (self.subsystem.options[0][1] if self.subsystem.options else None)
            self.subsystem.layout.display = ''
            self.variable.options = [(f'{m[0]} [{m[1]}]', k) for k, m in LOAD_META.items() if m[2] is not bool]
        else:
            self.subsystem.options = []
            self.subsystem.layout.display = 'none'
            self.variable.options = [(f'{m[0]} [{m[1]}]', k) for k, m in PARAMS[self.group.value].items()]
        choices = [value for _, value in self.variable.options]
        self.variable.value = old if old in choices else (choices[0] if choices else None)
        self.busy = False
        self.update_sweep_parameter()

    def sweep_path(self):
        if not self.variable.value:
            raise ValueError('Choose a numeric input.')
        if self.group.value != 'Loads':
            return 'parameters.' + self.variable.value
        if not self.subsystem.value:
            raise ValueError('The selected variants have no common subsystem.')
        prefix = 'loads' if self.sweep_mode.value == 'nominal' else 'safe_overrides'
        return f'{prefix}.{self.subsystem.value}.{self.variable.value}'

    def update_sweep_parameter(self):
        if self.busy:
            return
        self.busy = True
        field, is_load = self.variable.value, self.group.value == 'Loads'
        for name, control in [('orbit', self.link_orbit), ('duty', self.link_duty), ('peak', self.link_peak)]:
            applicable = ((not is_load and field in ('eclipse_h', 'sunlight_h')) if name == 'orbit' else
                          (is_load and self.subsystem.value in ('dsn', 'crosslink', 'broadcast') and field in ('duty_e', 'duty_d')) if name == 'duty' else
                          (is_load and field == 'active_W'))
            control.value, control.disabled = False, not applicable
            control.layout.display = '' if applicable else 'none'
        self.busy = False
        self.update_sweep_hint()
        self.show_links()
        self.invalidate('sweep')

    def sweep_meta(self):
        return (LOAD_META if self.group.value == 'Loads' else PARAM_META)[self.variable.value]

    def sweep_references(self):
        path, meta = self.sweep_path(), self.sweep_meta()
        parts = path.split('.')
        references = {}
        for name in check_values(self.sweep_names):
            case = self.variants[name]
            value = case['parameters'][parts[1]] if len(parts) == 2 else mode_loads(case, self.sweep_mode.value)[parts[1]][parts[2]]
            references[name] = None if value is None else display_value(value, meta)
        return references

    def update_sweep_hint(self):
        try:
            meta, references = self.sweep_meta(), self.sweep_references()
        except (KeyError, ValueError):
            self.sweep_hint.value = 'Choose a subsystem shared by the selected variants.'
            return
        reference_text = ' · '.join(f'{html.escape(name)}: {"TBD" if value is None else fmt_number(value)} {meta[1]}' for name, value in references.items())
        instruction = 'Battery counts use unique integers; a range is rounded to discrete counts.' if meta[2] is int else 'Percentages use 0–100.' if meta[3] else 'Use the units shown in the variable label.'
        extra = ' Active power above the saved peak requires the explicit peak link below, or a larger peak in the saved variant.' if self.group.value == 'Loads' and self.variable.value == 'active_W' else ''
        self.sweep_hint.value = f'<small>Saved reference values: {reference_text or "choose a variant"}.<br>{instruction}{extra}</small>'

    def suggest_range(self):
        meta = self.sweep_meta()
        values = [v for v in self.sweep_references().values() if v is not None]
        ref = values[0] if values else (80 if self.variable.value == 'battery_capacity_retention' else 1)
        lo, hi = (max(1, int(ref)-2), int(ref)+2) if meta[2] is int else (max(0, ref*.75), ref*1.25 if ref else (10 if meta[3] else 1))
        if meta[3] and meta[5] == 1:
            hi = min(100, hi)
            lo = min(lo, hi*.75)
        if self.variable.value in ('annual_degradation', 'incidence_deg'):
            hi = min(99 if self.variable.value == 'annual_degradation' else 89, hi)
        if meta[4] == 0 and lo == 0 and self.variable.value not in ('eclipse_h', 'lifetime_years', 'load_margin', 'annual_degradation', 'incidence_deg', *LOAD_META):
            lo = max(.001, hi*.1)
        self.minimum.value, self.maximum.value = lo, hi
        self.points.value = min(11, int(hi-lo+1)) if meta[2] is int else 11
        self.range_style.value = 'Range'
        self.message('Suggested a range from the first selected variant. Review links and bounds before running.', area=2)

    def selected_links(self):
        return tuple(name for name, control in [('orbit', self.link_orbit), ('duty', self.link_duty), ('peak', self.link_peak)] if not control.disabled and control.value)

    def show_links(self):
        labels = {'orbit': 'hold orbit period', 'duty': 'equal communication duties', 'peak': 'peak = active'}
        self.links_label.value = '<small><b>Active links:</b> ' + (', '.join(labels[k] for k in self.selected_links()) or 'none') + '</small>'

    def links_changed(self):
        if not self.busy:
            self.show_links()
            self.invalidate('sweep')

    def sweep_values(self):
        if self.range_style.value == 'Explicit list':
            try:
                values = np.asarray([float(value.strip()) for value in self.explicit.value.split(',')])
            except ValueError:
                raise ValueError('Enter comma-separated numbers, for example: 10, 20, 30. Do not include units or a trailing comma.') from None
        else:
            if not 2 <= self.points.value <= 500:
                raise ValueError('Choose 2–500 points.')
            if not np.isfinite([self.minimum.value, self.maximum.value]).all() or not self.minimum.value < self.maximum.value:
                raise ValueError('Minimum and maximum must be finite, with minimum less than maximum.')
            if self.sweep_meta()[2] is int:
                if self.minimum.value < 1 or any(int(v) != v for v in (self.minimum.value, self.maximum.value)):
                    raise ValueError('Battery count range endpoints must be positive integers.')
                values = np.unique(np.rint(np.linspace(self.minimum.value, self.maximum.value, self.points.value))).astype(int)
            else:
                values = np.linspace(self.minimum.value, self.maximum.value, self.points.value)
        if len(values) > 500 or not np.isfinite(values).all():
            raise ValueError('Use at most 500 finite numeric values.')
        return np.unique(values)

    def run_sweep(self):
        self.sweep_fresh = False
        self.sweep_out.children = ()
        self.update_export()
        self.require_analysis_ready()
        names, outputs, mode = check_values(self.sweep_names), check_values(self.sweep_outputs), self.sweep_mode.value
        if not outputs:
            raise ValueError('Choose at least one output.')
        values, path, meta = self.sweep_values(), self.sweep_path(), self.sweep_meta()
        references, links = self.sweep_references(), self.selected_links()
        frame = sweep_cases(self.variants, names, mode, path, values, links)
        frame.insert(2, 'Input', meta[0])
        frame.insert(3, 'Input unit', meta[1])
        frame.insert(4, 'Input path', path)
        frame.insert(5, 'Links', ', '.join(links) or 'none')
        children = [self.note(f'{len(values)} points × {len(names)} variants · {mode}', 'Dotted vertical lines mark saved reference values within the range. All points validated before plotting.')]
        children.extend(result_charts(frame, outputs, mode, sweep=(meta, references)))
        table = table_widget(frame[['Variant', 'Mode', 'Input value', *outputs]])
        children.append(self.accordion(table, 'Sweep results table'))
        self.sweep_out.children = tuple(children)
        self.sweep_data = frame[['Variant', 'Mode', 'Input', 'Input unit', 'Input path', 'Links', 'Input value', *outputs]].copy()
        self.sweep_fresh = True
        self.update_export()
        self.message(f'Completed {len(values)} unique points per variant. Saved inputs were not modified.', area=2)


    @staticmethod
    def local_path(name, suffix):
        path = Path(name)
        if path.name != name or not name or path.suffix.lower() != suffix:
            raise ValueError(f'Use a simple {suffix} filename in the notebook working directory.')
        if path.is_symlink() or (path.exists() and not path.is_file()):
            raise ValueError('Choose a regular file, not a directory or symbolic link.')
        return path

    def confirm_write(self, path):
        token = str(path.resolve())
        if path.exists() and (self.pending_write != token or not self.overwrite.value):
            self.pending_write = token
            self.overwrite.value = False
            raise ValueError(f'{path.name} already exists. Select Confirm overwrite, then click the same write button again.')
        self.pending_write = None
        self.overwrite.value = False

    def update_export(self):
        if hasattr(self, 'export_button'):
            fresh = self.comparison_fresh if self.csv_source.value == 'Comparison' else self.sweep_fresh
            self.export_button.disabled = not fresh

    def save_json(self):
        self.require_clean()
        path = self.local_path(self.file_name.value, '.json')
        self.confirm_write(path)
        write_variants(path, self.variants)
        self.disk_dirty = False
        self.message(f'Saved {len(self.variants)-1} complete variants to {path}.')

    def load_json(self):
        self.require_clean()
        incoming = read_variants(self.local_path(self.file_name.value, '.json'))
        new, loaded = deepcopy(self.variants), []
        for name, case in incoming.items():
            unique, i = name, 2
            while unique in new:
                unique, i = f'{name} ({i})', i+1
            new[unique] = deepcopy(case)
            loaded.append(unique)
        self.variants = new
        self.refresh_names(select_new=loaded)
        self.message('Loaded: ' + (', '.join(loaded) or 'no variants in file') + '. Name conflicts receive a numeric suffix.')

    def export_csv(self):
        fresh = self.comparison_fresh if self.csv_source.value == 'Comparison' else self.sweep_fresh
        data = self.comparison_data if self.csv_source.value == 'Comparison' else self.sweep_data
        if not fresh or data is None:
            raise ValueError('Results are out of date. Update comparison or Run sweep before exporting.')
        path = self.local_path(self.csv_name.value, '.csv')
        self.confirm_write(path)
        atomic_text(path, data.to_csv(index=False, na_rep=''))
        self.message(f'Exported current {self.csv_source.value.lower()} results to {path}. Blank cells mean TBD or undefined.')


## Launch
The interface stores variants in memory until you export JSON. Rerunning this launch cell preserves the current tool; restarting the kernel resets the session. After changing the Python baseline or interface definitions, save variants and restart/run all to apply them.

Verified with 49 regression checks and a fresh-kernel Run All in JupyterLab, including real browser interactions for creating/editing variants, comparison, validation and overlaid sweeps. Baseline nominal/safe results match exactly. Classic Notebook and VS Code frontends were not checked.

In [5]:
if widgets is None:
    print("Interactive editor unavailable. Run %pip install ipywidgets, restart, then Run All.")
    print('Python fallback: table, results = calculate(baseline, "nominal")')
    display(pd.DataFrame({mode: calculate(baseline, mode)[1] for mode in ("nominal", "safe")}).loc[KEY_RESULTS].round(3).fillna("TBD"))
else:
    if "power_tool" not in globals():
        power_tool = PowerBudgetUI()
    display(power_tool.root)
